# 03_6. Test evaluation — the single opening of test / Оценка на test — единственное открытие test

**EN.** This notebook follows the frozen test protocol (`reports/modeling/test_protocol.md`) step by step and nothing else: every model, calibrator, LGD rule, alert threshold and lead-time definition was fixed on train and validation before. The first cell checks that the protocol is frozen and that the configs and model files have the sha256 written in it; if anything differs, the notebook stops. Anything changed after this run is post-hoc and goes to the last section of the protocol.

Task: Y1 (first 90+ days past due or a default exit within 12 months) for loans that are current at t. Contents: (1) predictions; (2) comparison table B0 / M1 / M2 with the XGBoost and LightGBM comparators, 95 % cluster-bootstrap CIs and pairwise DeLong tests; (3) the alert list at calibrated PD ≥ 0.01 (sensitivity 0.02 and 0.005) and the AUC inside FICO bands; (4) **H3**; (5) **H4**. Sensitivity checks and score stability are in `03_6b`. Test = observation dates 2018-03 … 2018-12, all loans of the 15 % sample (including the 20 % holdout loans), without disaster-affected rows.

**RU.** Ноутбук шаг за шагом выполняет замороженный протокол test (`reports/modeling/test_protocol.md`) и ничего сверх него: все модели, калибраторы, правило LGD, порог тревоги и определение lead time зафиксированы раньше на train и validation. Первая ячейка проверяет, что протокол заморожен и что у конфигов и файлов моделей те же sha256, что записаны в нём; при расхождении ноутбук останавливается. Всё, что меняется после этого запуска, — post-hoc и записывается в последний раздел протокола.

Задача: Y1 (первая просрочка 90+ дней или выход по дефолту за 12 месяцев) для кредитов без просрочки на дату t. Содержание: (1) прогнозы; (2) таблица сравнения B0 / M1 / M2 со сравнительными моделями XGBoost и LightGBM, 95 %-ми интервалами кластерного bootstrap и попарными тестами DeLong; (3) список тревог при откалиброванной PD ≥ 0,01 (чувствительность — 0,02 и 0,005) и AUC внутри групп FICO; (4) **H3**; (5) **H4**. Проверки чувствительности и стабильность скора — в `03_6b`. Test — даты наблюдения 2018-03 … 2018-12, все кредиты 15 %-й выборки (включая отложенные 20 %), без наблюдений, затронутых бедствиями.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import gc
import subprocess
import sys
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
pd.set_option("display.width", 230)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.dpi": 80, "axes.grid": True, "grid.alpha": 0.3})

from xai_ews.config import data_config, feature_config
from xai_ews.evaluation.compare import h4_check, metrics_table, pairwise
from xai_ews.evaluation.el import alert_mask, alert_threshold, assign_lgd, captured_el_share, captured_event_share
from xai_ews.evaluation.heterogeneity import auc_within_segments
from xai_ews.evaluation.leadtime import bootstrap_median, event_months, lead_times, paired_bootstrap, summarise
from xai_ews.evaluation.protocol import check_frozen
from xai_ews.models import registry
from xai_ews.models.data import EVENT_MONTH, load_eval, load_snapshots
from xai_ews.paths import MODELING_DIR, PREDICTIONS_DIR, PROJECT_ROOT

SPLIT = "test"               # "validation" only for the dry run of the code before the protocol is frozen
OUT = MODELING_DIR / ("test" if SPLIT == "test" else "dry_run_validation")
OUT.mkdir(parents=True, exist_ok=True)
if SPLIT == "test":
    print(check_frozen())    # stops here if the protocol is not frozen or a hash differs
SEED, B = 20261025, 1000
DCFG, SEL = data_config(), feature_config()["selected"]
def mem():
    import resource
    return f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9:.1f} GB peak"

TASK = "y1"
CATS_ALL = list(dict.fromkeys([*SEL["categorical"], *SEL["categorical_h3"]]))
FEATS_ALL = list(dict.fromkeys([*SEL["features"], *SEL["h3_full"]]))
LT_WINDOW = {"validation": ("2015-09", "2017-12", False), "test": ("2017-06", "2019-09", None)}[SPLIT]
MAIN = ["b0_y1", "m1_y1", "m2_y1"]
GBDT = ["xgb_y1", "lgb_y1"]           # comparators, scored in a process of their own (scripts/score_gbdt.py)
EXTRA = ["m1_y1_noomega", "h3_static_y1", "h3_full_y1", "m1_y1_linear", "m1_y1_r01", "m1_y1_noprepaid"]
THRESHOLDS = [alert_threshold(), *DCFG["alert"]["pd_threshold_sensitivity"]]     # main first
FICO = "borrower_credit_score_at_origination"

protocol frozen, 79 files verified: **Статус: ЗАМОРОЖЕН 2026-11-03 10:54** — второе открытие test (раздел 0). Протокол пишется до открытия test; пробный прогон кода 03_6 / 03_6b сделан на validation 2026-11-03 (все ячейки выполнены, результаты — `reports/modeling/dry_run_validation/`; в нём калибраторы оцениваются и на половине B, поэтому цифры — только проверка кода). Test открывается один раз ноутбуками `notebooks/03_6_test_evaluation.ipynb` и `notebooks/03_6b_test_sensitivity.ipynb`; их первая ячейка сверяет sha256 из раздела 12. Любое изменение после просмотра test помечается как post-hoc и записывается в раздел 13 и в журнал решений.


### 1. Predictions / Прогнозы

**EN.** All rows of the split are loaded once (with every disaster flag, so the sensitivity variants can re-filter them); every model gives its raw logit, and PD = its Platt calibrator from validation half B. XGBoost and LightGBM are scored first by `scripts/score_gbdt.py` in a separate process (their OpenMP runtime hangs next to PyTorch) and joined by (loan, month). LGD by the current MTM LTV segment. M1 and M2 test logits are saved for stage 4.

**RU.** Все строки выборки загружаются один раз (со всеми флагами бедствий, чтобы варианты чувствительности могли отфильтровать их по-своему); каждая модель даёт сырой логит, PD — через её калибратор Платта с половины B validation. XGBoost и LightGBM считаются первыми скриптом `scripts/score_gbdt.py` в отдельном процессе (их среда OpenMP зависает рядом с PyTorch) и присоединяются по (кредит, месяц). LGD — по сегменту текущего MTM LTV. Логиты M1 и M2 на test сохраняются для этапа 4.

In [2]:
# XGBoost / LightGBM first, in a separate process: with PyTorch (M2) in the same process they hang
subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "score_gbdt.py"), SPLIT], check=True, cwd=PROJECT_ROOT)
df = load_eval(SPLIT, TASK, FEATS_ALL, CATS_ALL, exclusion="all").sort("loan_identifier", "m")   # fixed row order -> reproducible bootstrap
cols = {}
for name in [*MAIN, *EXTRA]:
    z = registry.predict_any(name, df)
    cols[f"z_{name}"] = z.astype(np.float32)
    cols[f"pd_{name}"] = registry.calibrator(name).predict(z).astype(np.float32)
    gc.collect()
keep = ["loan_identifier", "m", TASK, EVENT_MONTH[TASK], "upb", FICO, "holdout_loan", "new_cohort", "prepaid_in_window",
        "excl_main", "excl_hurricane", "excl_any_overlap"]
frame = df.select(keep).with_columns(assign_lgd(df, DCFG), *[pl.Series(k, v) for k, v in cols.items()])
del df, cols; gc.collect()
frame = frame.join(pl.read_parquet(PREDICTIONS_DIR / f"gbdt_{SPLIT}.parquet"), on=["loan_identifier", "m"],
                   how="left", maintain_order="left")
assert all(frame[f"z_{n}"].null_count() == 0 for n in GBDT), "comparator logits missing for some rows"
frame = frame.with_columns(*[pl.Series(f"pd_{n}", registry.calibrator(n).predict(frame[f"z_{n}"].to_numpy()).astype(np.float32))
                             for n in GBDT])
for name in ("m1_y1", "m2_y1") if SPLIT == "test" else ():   # stage-4 input; never overwrite validation files
    frame.select("loan_identifier", "m", TASK, pl.col(f"z_{name}").alias("logit")).write_parquet(
        PREDICTIONS_DIR / f"{name}_{SPLIT}.parquet")
frame.write_parquet(PREDICTIONS_DIR / f"{SPLIT}_frame_{TASK}.parquet")   # read by 03_6b
main = frame.filter(~pl.col("excl_main"))
y, loans = main[TASK].to_numpy(), main["loan_identifier"].to_numpy()
lgd, ead = main["lgd"].to_numpy(), main["upb"].to_numpy()
print(frame.height, "rows,", "main sample", main.height, "rows,", int(y.sum()), "events | memory", mem())

/Users/zarinanurzanova/Desktop/xai-early-credit-risk-detection/src/xai_ews/models/gbdt.py:75: UserWarning: [10:54:38] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1509: Unknown file format: `model`. Using UBJSON (`ubj`) as a guess.
  b.load_model(str(path))


/Users/zarinanurzanova/Desktop/xai-early-credit-risk-detection/src/xai_ews/models/gbdt.py:75: UserWarning: [10:54:40] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1509: Unknown file format: `model`. Using UBJSON (`ubj`) as a guess.
  b.load_model(str(path))


/Users/zarinanurzanova/Desktop/xai-early-credit-risk-detection/src/xai_ews/models/gbdt.py:75: UserWarning: [10:54:42] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1509: Unknown file format: `model`. Using UBJSON (`ubj`) as a guess.
  b.load_model(str(path))


/Users/zarinanurzanova/Desktop/xai-early-credit-risk-detection/src/xai_ews/models/gbdt.py:75: UserWarning: [10:54:44] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1509: Unknown file format: `model`. Using UBJSON (`ubj`) as a guess.
  b.load_model(str(path))


/Users/zarinanurzanova/Desktop/xai-early-credit-risk-detection/src/xai_ews/models/gbdt.py:75: UserWarning: [10:54:46] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1509: Unknown file format: `model`. Using UBJSON (`ubj`) as a guess.
  b.load_model(str(path))


/Users/zarinanurzanova/Desktop/xai-early-credit-risk-detection/src/xai_ews/models/gbdt.py:75: UserWarning: [10:55:05] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1509: Unknown file format: `model`. Using UBJSON (`ubj`) as a guess.
  b.load_model(str(path))


... (12 outputs left out)

/Users/zarinanurzanova/Desktop/xai-early-credit-risk-detection/src/xai_ews/models/gbdt.py:75: UserWarning: [10:56:23] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1509: Unknown file format: `model`. Using UBJSON (`ubj`) as a guess.
  b.load_model(str(path))


/Users/zarinanurzanova/Desktop/xai-early-credit-risk-detection/src/xai_ews/models/gbdt.py:75: UserWarning: [10:56:25] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1509: Unknown file format: `model`. Using UBJSON (`ubj`) as a guess.
  b.load_model(str(path))


test 4111592 rows scored by xgb_y1, lgb_y1


4111592 rows, main sample 3758886 rows, 7824 events | memory 1.7 GB peak


### 2. Comparison table / Таблица сравнения

**EN.** Metrics of the methodology on the main sample. `*_lo` / `*_hi` = 95 % cluster-bootstrap CI by loan (B = 1000). Alert list = calibrated PD ≥ 0.01: `alert_share` (share of rows alerted), `alert_precision` (event rate on the list), `alert_recall` (share of events on the list), `el_share_alert` (share of realised EL on the list); `auc_upb_weighted` = AUC with rows weighted by the balance. Then pairwise ΔAUC with DeLong on one random snapshot per loan, Holm's correction over the three main pairs (M1 − B0, M1 − M2, M2 − B0) and, separately, over the two comparator pairs (M1 − XGBoost, M1 − LightGBM; descriptive).

**RU.** Метрики методологии на основной выборке. `*_lo` / `*_hi` — 95 %-й интервал кластерного bootstrap по кредиту (B = 1000). Список тревог — откалиброванная PD ≥ 0,01: `alert_share` (доля строк под тревогой), `alert_precision` (доля событий в списке), `alert_recall` (доля событий, попавших в список), `el_share_alert` (доля фактического EL в списке); `auc_upb_weighted` — AUC, где строки взвешены по ОД. Затем попарные ΔAUC с DeLong на одном случайном снимке на кредит и поправкой Холма по трём основным парам (M1 − B0, M1 − M2, M2 − B0) и отдельно по двум парам сравнительных моделей (M1 − XGBoost, M1 − LightGBM; описательно).

In [3]:
comparison = metrics_table(y, {n: main[f"pd_{n}"].to_numpy() for n in [*MAIN, *GBDT]}, loans, lgd, ead,
                           threshold=alert_threshold(), b=B, seed=SEED)
comparison.to_csv(OUT / "comparison_table.csv")
z_of = lambda names: {n: main[f"z_{n}"].to_numpy() for n in names}
b0, m1, m2 = MAIN
pw = pd.concat({"main": pairwise(y, z_of(MAIN), loans, [(m1, b0), (m1, m2), (m2, b0)], b=B, seed=SEED),
                "comparators": pairwise(y, z_of([m1, *GBDT]), loans, [(m1, g) for g in GBDT], b=B, seed=SEED)},
               names=["family"])
pw.to_csv(OUT / "comparison_pairwise.csv")
display(comparison[["n", "events", "event_rate", "mean_pd", "auc", "auc_lo", "auc_hi", "ks", "pr_auc", "brier", "ece_norm",
                    "cal_slope", "cal_intercept", "auc_upb_weighted", "alert_share", "alert_precision", "alert_recall",
                    "el_share_alert", "el_share_alert_lo", "el_share_alert_hi"]])
pw

,n,events,event_rate,mean_pd,auc,auc_lo,auc_hi,ks,pr_auc,brier,ece_norm,cal_slope,cal_intercept,auc_upb_weighted,alert_share,alert_precision,alert_recall,el_share_alert,el_share_alert_lo,el_share_alert_hi
model,,,,,,,,,,,,,,,,,,,,
b0_y1,3758886,7824,0.0021,0.0019,0.8247,0.8186,0.8310,0.5029,0.0101,0.0021,0.0710,0.9355,0.0742,0.8280,0.0334,0.0141,0.2262,0.2091,0.1957,0.2221
m1_y1,3758886,7824,0.0021,0.0021,0.8378,0.8319,0.8436,0.5268,0.0126,0.0021,0.0623,0.9321,0.0049,0.8403,0.0371,0.0155,0.2763,0.2717,0.2565,0.2864
m2_y1,3758886,7824,0.0021,0.0021,0.8335,0.8275,0.8394,0.5211,0.0121,0.0021,0.0287,0.9570,0.0140,0.8365,0.0370,0.0152,0.2704,0.2708,0.2555,0.2859
xgb_y1,3758886,7824,0.0021,0.0021,0.8390,0.8331,0.8451,0.5324,0.0136,0.0021,0.0680,0.9443,-0.0042,0.8413,0.0421,0.0149,0.3020,0.3062,0.2902,0.3219
lgb_y1,3758886,7824,0.0021,0.0021,0.8387,0.8328,0.8449,0.5297,0.0133,0.0021,0.0777,0.9373,0.0058,0.8408,0.0394,0.0154,0.2914,0.2903,0.2750,0.3060


pair  delta_auc      lo     hi  share_le_0  delong_delta_one_snapshot  delong_p  one_snapshot_rows  one_snapshot_events  delong_p_holm
family                                                                                                                                                         
main        0   m1_y1 − b0_y1     0.0131  0.0107 0.0155      0.0000                     0.0134    0.0000            1096460                 3582         0.0000
            1   m1_y1 − m2_y1     0.0042  0.0022 0.0063      0.0000                     0.0042    0.0002            1096460                 3582         0.0002
            2   m2_y1 − b0_y1     0.0089  0.0065 0.0114      0.0000                     0.0093    0.0000            1096460                 3582         0.0000
comparators 0  m1_y1 − xgb_y1    -0.0012 -0.0028 0.0003      0.9360                    -0.0014    0.0640            1096460                 3582         0.1280
            1  m1_y1 − lgb_y1    -0.0009 -0.0022 0.0004      0.8990                    -0.0012    0.0762            1096460                 3582         0.1280

**EN. How to read.** On test (3,758,886 rows, 7,824 events, 0.21 %) the order is the same as on validation: B0 0.825 [0.819; 0.831] < M2 0.834 [0.828; 0.839] < M1 0.838 [0.832; 0.844] ≈ XGBoost 0.839 ≈ LightGBM 0.839. All three main differences are significant after Holm's correction: M1 − B0 +0.013 [0.011; 0.015], M1 − M2 +0.004 [0.002; 0.006], M2 − B0 +0.009 [0.006; 0.011]. M1 − XGBoost −0.0012 [−0.0028; +0.0003] and M1 − LightGBM −0.0009 [−0.0022; +0.0004] (Holm p = 0.13): the three boosting libraries are equal, as on validation. AUC is about 0.02 lower than on validation (M1 0.857) for every model alike. **The PD level holds**: mean PD of M1 0.207 % against the observed 0.208 %, calibration intercept +0.005; slope 0.93 (< 1: the spread of PD is slightly too wide, the highest PDs are a little too high); M2 has the lowest ECE. On the alert list (PD ≥ 0.01) M1 flags 3.7 % of rows and catches 27.6 % of events and 27.2 % of the realised EL; B0 22.6 % / 20.9 %; XGBoost 30.2 % / 30.6 %, but with a longer list (4.2 % of rows).

**RU. Как читать.** На test (3 758 886 строк, 7 824 события, 0,21 %) порядок тот же, что на validation: B0 0,825 [0,819; 0,831] < M2 0,834 [0,828; 0,839] < M1 0,838 [0,832; 0,844] ≈ XGBoost 0,839 ≈ LightGBM 0,839. Все три основные разности значимы после поправки Холма: M1 − B0 +0,013 [0,011; 0,015], M1 − M2 +0,004 [0,002; 0,006], M2 − B0 +0,009 [0,006; 0,011]. M1 − XGBoost −0,0012 [−0,0028; +0,0003] и M1 − LightGBM −0,0009 [−0,0022; +0,0004] (p Холма = 0,13): три библиотеки бустинга равны, как и на validation. AUC примерно на 0,02 ниже, чем на validation (M1 0,857), у всех моделей одинаково. **Уровень PD держится**: средняя PD M1 0,207 % при фактической доле 0,208 %, calibration intercept +0,005; slope 0,93 (< 1: разброс PD чуть шире нужного, самые высокие PD немного завышены); наименьший ECE — у M2. В списке тревог (PD ≥ 0,01) M1 отмечает 3,7 % строк и ловит 27,6 % событий и 27,2 % фактического EL; B0 — 22,6 % / 20,9 %; XGBoost — 30,2 % / 30,6 %, но при более длинном списке (4,2 % строк).

### 3. Alert list at three thresholds and AUC inside FICO bands / Список тревог при трёх порогах и AUC внутри групп FICO

**EN.** Point values for each threshold (0.01 main; 0.02 and 0.005 sensitivity): alerts, share of rows, precision, share of events and of realised EL caught. Then the AUC of each model inside the FICO bands < 680 (incl. missing) / 680–739 / ≥ 740: how well the model ranks borrowers with similar credit scores.

**RU.** Точечные значения для каждого порога (0,01 — основной; 0,02 и 0,005 — чувствительность): число тревог, доля строк, точность, доля пойманных событий и фактического EL. Затем AUC каждой модели внутри групп FICO < 680 (с пропусками) / 680–739 / ≥ 740: насколько хорошо модель различает заёмщиков с похожим кредитным баллом.

In [4]:
rows = []
for thr in THRESHOLDS:
    for n in [*MAIN, *GBDT, "h3_static_y1", "h3_full_y1"]:
        al = alert_mask(main[f"pd_{n}"].to_numpy(), thr)
        rows.append({"threshold": thr, "model": n, "alerts": int(al.sum()), "alert_share": float(al.mean()),
                     "precision": float(y[al].mean()) if al.any() else np.nan,
                     "event_share": captured_event_share(y, al), "el_share": captured_el_share(y, al, lgd, ead)})
alerts = pd.DataFrame(rows)
alerts.to_csv(OUT / "alert_capture.csv", index=False)
within = pd.concat([auc_within_segments(y, main[f"z_{n}"].to_numpy(), main[FICO].to_numpy()).assign(model=n)
                    for n in [*MAIN, *GBDT, "h3_static_y1", "h3_full_y1"]], ignore_index=True)
within.to_csv(OUT / "auc_within_fico.csv", index=False)
display(alerts.set_index(["threshold", "model"]))
within.pivot(index="model", columns="segment", values="auc")[["<680", "680-739", ">=740"]]

alerts  alert_share  precision  event_share  el_share
threshold model                                                              
0.0100    b0_y1         125582       0.0334     0.0141       0.2262    0.2091
          m1_y1         139474       0.0371     0.0155       0.2763    0.2717
          m2_y1         139120       0.0370     0.0152       0.2704    0.2708
          xgb_y1        158104       0.0421     0.0149       0.3020    0.3062
          lgb_y1        148277       0.0394     0.0154       0.2914    0.2903
          h3_static_y1  146641       0.0390     0.0132       0.2471    0.2397
          h3_full_y1    143423       0.0382     0.0154       0.2823    0.2826
0.0200    b0_y1          15722       0.0042     0.0192       0.0386    0.0317
          m1_y1          35296       0.0094     0.0241       0.1085    0.1090
          m2_y1          24318       0.0065     0.0246       0.0766    0.0812
          xgb_y1         36316       0.0097     0.0249       0.1155    0.1229
          lgb_y1         36824       0.0098     0.0242       0.1138    0.1160
          h3_static_y1   19669       0.0052     0.0197       0.0495    0.0493
          h3_full_y1     35960       0.0096     0.0241       0.1109    0.1121
0.0050    b0_y1         397293       0.1057     0.0098       0.4968    0.4749
          m1_y1         388738       0.1034     0.0105       0.5221    0.5114
          m2_y1         425559       0.1132     0.0099       0.5381    0.5296
          xgb_y1        410183       0.1091     0.0103       0.5401    0.5349
          lgb_y1        393050       0.1046     0.0105       0.5253    0.5123
          h3_static_y1  429658       0.1143     0.0095       0.5202    0.5114
          h3_full_y1    396035       0.1054     0.0106       0.5343    0.5270

segment,<680,680-739,>=740
model,,,
b0_y1,0.6462,0.7172,0.7775
h3_full_y1,0.6898,0.7467,0.7969
h3_static_y1,0.6419,0.7183,0.7746
lgb_y1,0.6971,0.7430,0.7957
m1_y1,0.6903,0.7438,0.7945
m2_y1,0.6851,0.7348,0.7858
xgb_y1,0.7002,0.7436,0.7936


**EN. How to read.** At PD ≥ 0.02 the lists shrink to about 1 % of rows (M1 catches 10.9 % of events), at 0.005 they grow to about 10 % of rows (M1 52 %). Precision of M1's main list is 1.55 %, 7.4 times the base rate of 0.21 %. B0 catches the least at every threshold; at 0.02 B0 and the static H3 model flag only 0.4–0.5 % of rows (their PDs seldom reach that level). Inside FICO bands AUC is lower (0.64–0.80), because the credit score no longer separates the borrowers. **M1 gains most over B0 in the weakest band**: < 680 0.690 vs 0.646 (+0.044), 680–739 +0.027, ≥ 740 +0.017 — the behaviour and the interactions add most where FICO says least. Point values, no CIs.

**RU. Как читать.** При PD ≥ 0,02 списки сжимаются примерно до 1 % строк (M1 ловит 10,9 % событий), при 0,005 растут примерно до 10 % строк (M1 — 52 %). Точность основного списка M1 — 1,55 %, в 7,4 раза выше базовой доли 0,21 %. Меньше всех на каждом пороге ловит B0; при 0,02 B0 и статическая модель H3 отмечают лишь 0,4–0,5 % строк (их PD редко доходят до этого уровня). Внутри групп FICO AUC ниже (0,64–0,80), потому что кредитный балл уже не разделяет заёмщиков. **Сильнее всего M1 выигрывает у B0 в самой слабой группе**: < 680 — 0,690 против 0,646 (+0,044), 680–739 — +0,027, ≥ 740 — +0,017: поведение и взаимодействия признаков дают больше всего там, где FICO говорит меньше всего. Точечные значения, без интервалов.

### 4. H3 — hidden deterioration is detectable before delinquency / H3 — скрытое ухудшение обнаружимо до просрочки

**EN.** Decision rule (protocol): on Y1, the CatBoost on the full H3 set (static + dynamic, 23 features) vs the CatBoost on the static set (16) — ΔAUC > 0 with DeLong p < 0.05 (one random snapshot per loan) **and** the lower bound of the 95 % CI of the median lead time of the full model ≥ 3 months (alert = calibrated PD ≥ 0.01, quarterly snapshots in the 12 months before the event). Descriptive, fixed before test: the paired differences full − static of the share of event loans alerted and of the median lead time, for all three thresholds.

**RU.** Решающее правило (протокол): на Y1 CatBoost на полном наборе H3 (статические + динамические, 23 признака) против CatBoost на статическом наборе (16) — ΔAUC > 0 при p DeLong < 0,05 (один случайный снимок на кредит) **и** нижняя граница 95 %-го интервала медианного lead time полной модели ≥ 3 мес. (тревога — откалиброванная PD ≥ 0,01, квартальные снимки за 12 месяцев до события). Описательно, зафиксировано до test: парные разности «полная − статическая» доли кредитов с тревогой и медианного lead time для всех трёх порогов.

In [5]:
h3_auc = pairwise(y, z_of(["h3_full_y1", "h3_static_y1"]), loans, [("h3_full_y1", "h3_static_y1")], b=B, seed=SEED)
display(h3_auc)
ev = event_months(main, TASK, EVENT_MONTH[TASK])
lt_rows, lt_frames = [], {}
for name in ("h3_full_y1", "h3_static_y1", "m1_y1", "m2_y1", "b0_y1"):
    feats, cats = (SEL["features"], SEL["categorical"]) if name.startswith("b0_") else \
        (registry.meta(name)["features"], registry.meta(name)["categorical"])
    snap = load_snapshots(*LT_WINDOW[:2], TASK, feats, cats, holdout=LT_WINDOW[2])
    scored = snap.select("loan_identifier", "m").with_columns(
        pl.Series("pd", registry.calibrator(name).predict(registry.predict_any(name, snap)).astype(np.float32)))
    del snap; gc.collect()
    for thr in THRESHOLDS:
        lt = lead_times(ev, scored, thr)
        lt_frames[(name, thr)] = lt
        ci = bootstrap_median(lt, b=B, seed=SEED)
        lt_rows.append({"model": name, "threshold": thr, **summarise(lt), "median_lo": ci["lo"], "median_hi": ci["hi"]})
    del scored; gc.collect()
    print(name, "lead time done | memory", mem(), flush=True)
leadtime = pd.DataFrame(lt_rows)
leadtime.to_csv(OUT / "lead_time.csv", index=False)
thr0 = THRESHOLDS[0]
r = h3_auc.iloc[0]
lt_main = leadtime[(leadtime["model"] == "h3_full_y1") & (leadtime["threshold"] == thr0)].iloc[0]
h3_verdict = pd.Series({"delta_auc": r["delta_auc"], "delta_ci": f"[{r['lo']:.4f}; {r['hi']:.4f}]",
                        "delong_p": r["delong_p"], "auc_part": bool(r["delta_auc"] > 0 and r["delong_p"] < 0.05),
                        "median_lead": lt_main["median_lead"], "median_lead_ci": f"[{lt_main['median_lo']:.0f}; {lt_main['median_hi']:.0f}]",
                        "lead_part": bool(lt_main["median_lo"] >= 3)})
h3_verdict["H3"] = "confirmed" if h3_verdict["auc_part"] and h3_verdict["lead_part"] else (
    "partial" if h3_verdict["auc_part"] or h3_verdict["lead_part"] else "rejected")
h3_verdict.to_frame("H3").to_csv(OUT / "h3_verdict.csv")
display(leadtime.set_index(["threshold", "model"]))
# descriptive, fixed in the protocol before test: full − static on the same event loans
h3_paired = pd.concat({thr: paired_bootstrap(lt_frames[("h3_full_y1", thr)], lt_frames[("h3_static_y1", thr)], b=B, seed=SEED)
                       for thr in THRESHOLDS}, names=["threshold"])
h3_paired.to_csv(OUT / "h3_lead_time_full_vs_static.csv")
display(h3_paired)
h3_verdict.to_frame("H3")

,pair,delta_auc,lo,hi,share_le_0,delong_delta_one_snapshot,delong_p,one_snapshot_rows,one_snapshot_events,delong_p_holm
0,h3_full_y1 − h3_static_y1,0.0152,0.0130,0.0173,0.0000,0.0169,0.0000,1096460,3582,0.0000


h3_full_y1 lead time done | memory 1.7 GB peak


h3_static_y1 lead time done | memory 1.7 GB peak


m1_y1 lead time done | memory 1.7 GB peak


m2_y1 lead time done | memory 1.7 GB peak


b0_y1 lead time done | memory 2.1 GB peak


,,event_loans,alerted,alert_rate,median_lead,mean_lead,share_ge6,median_lead_missed0,median_lo,median_hi
threshold,model,,,,,,,,,
0.0100,h3_full_y1,4553,1625,0.3569,10.0000,9.1668,0.8677,0.0000,10.0000,10.0000
0.0200,h3_full_y1,4553,739,0.1623,10.0000,8.6225,0.7997,0.0000,9.0000,10.0000
0.0050,h3_full_y1,4553,2726,0.5987,10.0000,9.6196,0.9028,7.0000,10.0000,10.0000
0.0100,h3_static_y1,4553,1182,0.2596,11.0000,9.9010,0.9247,0.0000,10.0000,11.0000
0.0200,h3_static_y1,4553,264,0.0580,10.0000,9.5720,0.8939,0.0000,10.0000,10.0000
0.0050,h3_static_y1,4553,2450,0.5381,11.0000,10.0024,0.9327,6.0000,11.0000,11.0000
0.0100,m1_y1,4553,1572,0.3453,10.0000,9.1705,0.8670,0.0000,10.0000,10.0000
0.0200,m1_y1,4553,734,0.1612,9.0000,8.5313,0.8025,0.0000,9.0000,10.0000
0.0050,m1_y1,4553,2670,0.5864,10.0000,9.6311,0.9041,7.0000,10.0000,10.0000


delta      lo      hi
threshold                                    
0.0100    alert_rate   0.0973  0.0857  0.1081
          median_lead -1.0000 -1.0000  0.0000
0.0200    alert_rate   0.1043  0.0940  0.1138
          median_lead  0.0000 -1.0000  0.0000
0.0050    alert_rate   0.0606  0.0496  0.0716
          median_lead -1.0000 -1.0000 -1.0000

,H3
delta_auc,0.0152
delta_ci,[0.0130; 0.0173]
delong_p,0.0000
auc_part,True
median_lead,10.0000
median_lead_ci,[10; 10]
lead_part,True
H3,confirmed


**EN. How to read.** **H3 is confirmed.** ΔAUC full − static +0.0152 [0.0130; 0.0173], DeLong p ≈ 1.5·10⁻⁴²; the median lead time of the full model is 10 months, CI [10; 10], above 3. Descriptive: the full model alerts 35.7 % of the 4,553 event loans, the static one 26.0 % (+9.7 pp [8.6; 10.8]; +10.4 pp at 0.02, +6.1 pp at 0.005). The median lead of the alerted loans is 10 months for the full model and 11 for the static one (Δ −1 [−1; 0]): the static model alerts fewer loans, mostly those risky from origination, which are flagged from the first snapshot; the dynamic features add loans that deteriorate later. 87 % of the alerted event loans were flagged 6 or more months ahead. Two caveats: lead time is measured on quarterly snapshots, so it takes few distinct values and the CI of the median is degenerate; and it counts only alerted loans — with the missed ones counted as 0 the median at PD ≥ 0.01 is 0, because fewer than half of the event loans are alerted.

**RU. Как читать.** **H3 подтверждена.** ΔAUC «полная − статическая» +0,0152 [0,0130; 0,0173], p DeLong ≈ 1,5·10⁻⁴²; медианный lead time полной модели — 10 месяцев, интервал [10; 10], выше 3. Описательно: полная модель поднимает тревогу по 35,7 % из 4 553 кредитов с событием, статическая — по 26,0 % (+9,7 п.п. [8,6; 10,8]; +10,4 п.п. при 0,02, +6,1 п.п. при 0,005). Медианное опережение у кредитов с тревогой — 10 месяцев у полной модели и 11 у статической (Δ −1 [−1; 0]): статическая отмечает меньше кредитов, в основном рискованные с момента выдачи, а они отмечены с первого же снимка; динамические признаки добавляют кредиты, которые ухудшаются позже. 87 % отмеченных кредитов с событием отмечены за 6 и более месяцев. Две оговорки: lead time считается по квартальным снимкам, поэтому принимает немного разных значений, и интервал медианы вырожден; и он считается только по кредитам с тревогой — если пропущенным приписать 0, медиана при PD ≥ 0,01 равна 0, потому что тревога есть меньше чем у половины кредитов с событием.

### 5. H4 — accounting for the balance raises the captured loss without hurting calibration / H4 — учёт ОД повышает долю пойманного убытка без потери калибровки

**EN.** Decision rule (protocol): M1 trained with the exposure weight ω vs the same CatBoost (same hyperparameters and trees) without ω, both Platt-calibrated on validation half B. (a) 95 % CI of Δ captured EL share on the alert list (each model's own list, calibrated PD ≥ 0.01) above 0; (b) |Δ calibration slope| ≤ 0.1 and |Δ intercept| ≤ 0.1 on test. Absolute calibration is descriptive.

**RU.** Решающее правило (протокол): M1, обученная с весом по ОД ω, против того же CatBoost (те же гиперпараметры и число деревьев) без ω; обе откалиброваны Платтом на половине B validation. (а) 95 %-й интервал Δ доли пойманного EL в списке тревог (у каждой модели свой список, откалиброванная PD ≥ 0,01) выше 0; (б) |Δ calibration slope| ≤ 0,1 и |Δ intercept| ≤ 0,1 на test. Абсолютная калибровка — описательно.

In [6]:
h4 = h4_check(main, TASK, "pd_m1_y1", "pd_m1_y1_noomega", threshold=alert_threshold(), b=B, seed=SEED)
pd.Series(h4).to_frame("H4").to_csv(OUT / "h4_verdict.csv")
pd.Series(h4).to_frame("H4")

,H4
el_share_with,0.2717
el_share_without,0.2766
delta,-0.0049
delta_lo,-0.0087
delta_hi,-0.0011
slope_with,0.9321
slope_without,0.9304
d_slope,0.0017
intercept_with,0.0049
intercept_without,0.0038


**EN. How to read.** **H4 is partial.** The calibration part holds (|Δ slope| 0.002, |Δ intercept| 0.001, both ≤ 0.1), the EL part does not: the list of M1 with ω catches 27.2 % of the realised EL, the same model without ω 27.7 %, Δ −0.5 pp [−0.87; −0.11] — the weight slightly *lowers* the captured EL. AUC is unchanged (0.8378 vs 0.8381). Validation half A showed no gain either (Δ +0.3 pp [−1.0; +1.6], CI includes 0). So the exposure weight does not move larger loans onto the alert list at this threshold; which loans are on the list is decided by the PD ranking. The sensitivity of this result (other thresholds, LGD, calibration) is in `03_6b`.

**RU. Как читать.** **H4 подтверждена частично.** Часть про калибровку выполнена (|Δ slope| 0,002, |Δ intercept| 0,001, обе ≤ 0,1), часть про EL — нет: список M1 с ω ловит 27,2 % фактического EL, та же модель без ω — 27,7 %, Δ −0,5 п.п. [−0,87; −0,11]: вес даже немного *снижает* пойманный EL. AUC не меняется (0,8378 против 0,8381). На половине A validation прироста тоже не было (Δ +0,3 п.п. [−1,0; +1,6], интервал включает 0). Значит, вес по ОД не переносит крупные кредиты в список тревог при этом пороге; кто попадает в список, решает ранжирование по PD. Чувствительность этого вывода (другие пороги, LGD, калибровка) — в `03_6b`.

**EN.** Sensitivity checks (protocol) and score stability are in `03_6b_test_sensitivity.ipynb`: it reads the test predictions saved above (`data/processed/predictions/test_frame_y1.parquet`) in a fresh process, so the run fits into 8 GB of memory.

**RU.** Проверки чувствительности (протокол) и стабильность скора — в `03_6b_test_sensitivity.ipynb`: он читает сохранённые выше прогнозы test (`data/processed/predictions/test_frame_y1.parquet`) в отдельном процессе, чтобы расчёт поместился в 8 ГБ памяти.

**EN. Summary.** Test was opened under the frozen protocol (its second opening, named in section 0 of the protocol); nothing was changed after the run. Y1, 3.76 M rows, 7,824 events. (1) M1 (CatBoost) AUC 0.838 beats the scorecard B0 by +0.013 and the MLP M2 by +0.004, both significant after Holm; XGBoost and LightGBM are equal to M1. (2) The calibrated PD level is right (0.207 % vs 0.208 %). (3) The alert list at PD ≥ 0.01 covers 3.7 % of rows and catches 27.6 % of events and 27.2 % of the realised EL; M1's gain over B0 is largest for FICO < 680. (4) **H3 confirmed**: the dynamic features add +0.015 AUC and alert 9.7 pp more event loans; median lead 10 months. (5) **H4 partial**: ω keeps the calibration but does not raise the captured EL (Δ −0.5 pp). AUC of all models is about 0.02 lower than on validation.

**RU. Итог.** Test открыт по замороженному протоколу (второе открытие, названо в разделе 0 протокола); после запуска ничего не менялось. Y1, 3,76 млн строк, 7 824 события. (1) M1 (CatBoost), AUC 0,838, превосходит скоринговую карту B0 на +0,013 и MLP M2 на +0,004, обе разности значимы после поправки Холма; XGBoost и LightGBM равны M1. (2) Уровень откалиброванной PD верный (0,207 % против 0,208 %). (3) Список тревог при PD ≥ 0,01 — 3,7 % строк, ловит 27,6 % событий и 27,2 % фактического EL; выигрыш M1 над B0 больше всего при FICO < 680. (4) **H3 подтверждена**: динамические признаки добавляют +0,015 AUC и на 9,7 п.п. больше кредитов с событием под тревогой; медианное опережение — 10 месяцев. (5) **H4 — частично**: ω сохраняет калибровку, но не повышает пойманный EL (Δ −0,5 п.п.). AUC всех моделей примерно на 0,02 ниже, чем на validation.